<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Preprocessing" data-toc-modified-id="Preprocessing-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Preprocessing</a></span><ul class="toc-item"><li><span><a href="#Train-Test-split" data-toc-modified-id="Train-Test-split-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Train-Test split</a></span></li><li><span><a href="#Cleaning---Encoding---Scaling" data-toc-modified-id="Cleaning---Encoding---Scaling-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Cleaning - Encoding - Scaling</a></span></li><li><span><a href="#Preprocessing-function" data-toc-modified-id="Preprocessing-function-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Preprocessing function</a></span></li></ul></li><li><span><a href="#Procédure-d'évaluation" data-toc-modified-id="Procédure-d'évaluation-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Procédure d'évaluation</a></span></li><li><span><a href="#Modélisation" data-toc-modified-id="Modélisation-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Modélisation</a></span><ul class="toc-item"><li><span><a href="#LinearRegression" data-toc-modified-id="LinearRegression-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>LinearRegression</a></span><ul class="toc-item"><li><span><a href="#First-Iteration-(Dropna-+-Onehot-encoding-of-ocean)" data-toc-modified-id="First-Iteration-(Dropna-+-Onehot-encoding-of-ocean)-3.1.1"><span class="toc-item-num">3.1.1&nbsp;&nbsp;</span>First Iteration (Dropna + Onehot encoding of ocean)</a></span></li><li><span><a href="#Second-Iteration-:-First-iteration-+-log-scale-of-skewed-columns" data-toc-modified-id="Second-Iteration-:-First-iteration-+-log-scale-of-skewed-columns-3.1.2"><span class="toc-item-num">3.1.2&nbsp;&nbsp;</span>Second Iteration : First iteration + log scale of skewed columns</a></span></li><li><span><a href="#Third-Iteration-:-Second-Iteration-+-StandardScaler" data-toc-modified-id="Third-Iteration-:-Second-Iteration-+-StandardScaler-3.1.3"><span class="toc-item-num">3.1.3&nbsp;&nbsp;</span>Third Iteration : Second Iteration + StandardScaler</a></span></li></ul></li><li><span><a href="#KNN" data-toc-modified-id="KNN-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>KNN</a></span></li></ul></li></ul></div>

# Preprocessing

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Data manip
import pandas as pd
import numpy as np
#import pandas_profiling

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler, FunctionTransformer
from sklearn.metrics import r2_score,max_error,mean_absolute_error, mean_squared_error

In [2]:
data = pd.read_csv("data/data_init.txt")
df1 = data.copy()
df1=df1.drop(["Unnamed: 0"], axis=1, errors='ignore')
df1=df1.dropna(axis=0)

## Train-Test split

In [4]:
X= df1.drop("median_house_value",axis=1)
y= df1["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

## Cleaning - Encoding - Scaling

<u>**Missing Values**<u/>

In [ ]:
def imputation (trainset,testset):
    trainset = trainset.dropna(axis=0)
    testset = testset.dropna(axis=0)
    
    print(trainset.shape)
    print(testset.shape)
    return trainset,testset

In [4]:
def imputation(df):
    df = df.dropna(axis=0)
    return df


imputer = Pipeline(

  steps=[

    ('imput', FunctionTransformer(imputation))

  ]

)

<u>**Ocean-proximity OneHotEncoder**<u/>

In [5]:
one_hot_encoder = Pipeline(steps=[('one_hot', OneHotEncoder(handle_unknown='ignore'))])

In [ ]:
def ohe(trainset,testset):
    
    # Apply one-hot encoder to each column with categorical data
    OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    OH_encoder.fit(trainset.select_dtypes("object"))
    OH_cols_train = pd.DataFrame(OH_encoder.transform(trainset.select_dtypes("object")))    
    OH_cols_test = pd.DataFrame(OH_encoder.transform(testset.select_dtypes("object")))

    # One-hot encoding removed index; put it back
    OH_cols_train.index = trainset.index
    OH_cols_test.index = testset.index

    # Remove categorical columns (will replace with one-hot encoding)
    num_trainset = trainset.drop(trainset.select_dtypes("object"), axis=1)
    num_testset = testset.drop(testset.select_dtypes("object"), axis=1)

    # Add one-hot encoded columns to numerical features
    OH_trainset = pd.concat([num_trainset, OH_cols_train], axis=1)
    OH_testset = pd.concat([num_testset, OH_cols_test], axis=1)
    
    print(OH_trainset.shape)
    print(OH_testset.shape)
    return OH_trainset, OH_trainset

<u>**Logarithm scale**<u/>

In [6]:
def log_scale(df):
    df = np.log(df)
    return df

log_scaler = Pipeline(

  steps=[

    ('log', FunctionTransformer(log_scale))

  ]

)

In [ ]:
def log_scale(trainset,testset):
    trainset[['total_rooms',"total_bedrooms", "population", "households"]] = np.log(trainset[['total_rooms',"total_bedrooms", "population", "households"]])
    testset[['total_rooms',"total_bedrooms", "population", "households"]] = np.log(testset[['total_rooms',"total_bedrooms", "population", "households"]])
    
    print(trainset.shape)
    print(testset.shape)
    return trainset,testset
log_scale(trainset,testset)

<u>**Standard scale**<u/>

In [8]:
standard_scaler = Pipeline(steps=[('ss', StandardScaler())])

In [ ]:
def standard(trainset, testset):
    ss = StandardScaler()
    ss.fit(trainset[['total_rooms',"total_bedrooms", "population", "households","median_income","housing_median_age"]])
    trainset[['total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"]] = ss.transform(trainset[['total_rooms',"total_bedrooms", "population", "households","median_income","housing_median_age"]])
    testset[['total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"]] = ss.transform(testset[['total_rooms',"total_bedrooms", "population", "households","median_income","housing_median_age"]])
    
    print(trainset.shape)
    print(testset.shape)
    return trainset, testset

standard(trainset, testset)

In [9]:
preprocessor = ColumnTransformer(

  transformers=[
    #('imput', imputer, ["longitude","latitude",'total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"]),
    ('encoding', one_hot_encoder, ['ocean_proximity']),
    ('log', log_scaler, ['total_rooms',"total_bedrooms", "population", "households"]),
    ("ss", standard_scaler, ['total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"])
  ],
)

## Preprocessing function

In [ ]:
def preprocessing(trainset, testset):
    trainset, testset = imputation(trainset, testset)
    print(trainset.shape)
    print(testset.shape)
    trainset, testset = ohe(trainset, testset)
    print(trainset.shape)
    print(testset.shape)
    trainset, testset = log_scale(trainset,testset)
    print(trainset.shape)
    print(testset.shape)
    trainset, testset = standard(trainset, testset)
    print(trainset.shape)
    print(testset.shape)
    
    X_train=trainset.drop("median_house_value", axis=1)
    y_train=trainset["median_house_value"]

    X_test=testset.drop("median_house_value", axis=1)
    y_test=testset["median_house_value"]
    
    return X_train, y_train, X_test, y_test

preprocessing(trainset, testset)

In [ ]:
trainset, testset = train_test_split(df1, test_size=0.3, random_state=0)

In [ ]:
testset.shape

In [ ]:
X_train, y_train, X_test, y_test = preprocessing(trainset, testset)

In [16]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(11435, 9)
(11435,)
(4901, 9)
(4901,)


# Procédure d'évaluation

In [ ]:
def evaluation(model):
    model.fit(X_train,y_train)
    ypred = model.predict(X_test)
    
    print("R²:", r2_score(y_test, ypred))
    print("ME:", max_error(y_test, ypred))
    print("MAE:", mean_absolute_error(y_test, ypred))
    print("MSE:", mean_squared_error(y_test, ypred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, ypred)))

# Modélisation

## LinearRegression

In [10]:
lr = Pipeline(

  steps=[

    ('preprocessor', preprocessor),

    ('lreg', LinearRegression())

  ]

)

In [11]:
lr.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('encoding',
                                                  Pipeline(steps=[('one_hot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ocean_proximity']),
                                                 ('log',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<function log_scale at 0x000002A76DC21EE0>))]),
                                                  ['total_rooms',
                                                   'total_bedrooms',
                                                   'population',
                                                   'households']),
                                                 ('ss',
                                                  P

In [12]:
y_pred=lr.predict(X_test)
print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R²: 0.6598642857394634
MAE: 49252.58702305652


In [13]:
knn = Pipeline(

  steps=[

    ('preprocessor', preprocessor),

    ('knn', KNeighborsRegressor(n_neighbors = 5))

  ]

)

In [14]:
knn.fit(X_train,y_train)
y_pred=knn.predict(X_test)
print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R²: 0.6883051594575536
MAE: 45315.05419302183


In [19]:
X_train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
6649,-121.66,37.93,19.0,2055.0,358.0,1064.0,350.0,4.7426,INLAND
16218,-117.82,33.67,17.0,2895.0,439.0,1588.0,450.0,6.2760,<1H OCEAN
14066,-121.76,37.70,9.0,3980.0,736.0,1705.0,679.0,5.7068,INLAND
1361,-118.55,34.04,41.0,1482.0,239.0,617.0,242.0,8.8619,<1H OCEAN
15960,-118.30,33.77,18.0,3821.0,795.0,2831.0,769.0,2.9571,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
13268,-119.29,36.31,14.0,2382.0,377.0,1278.0,386.0,5.1896,INLAND
3297,-122.41,37.60,31.0,4424.0,834.0,1915.0,817.0,4.1364,NEAR OCEAN
9957,-119.61,36.31,25.0,1847.0,371.0,1460.0,353.0,1.8839,INLAND
10921,-117.23,33.78,23.0,3465.0,703.0,2672.0,607.0,1.9767,INLAND


In [15]:
errors = []
for k in range(2,15):
    model2 = KNeighborsRegressor(n_neighbors = k)
    errors.append(100*(1 - model2.fit(X_train, y_train).score(X_test, y_test)))
plt.plot(range(2,15), errors, 'o-')
plt.show()

ValueError: could not convert string to float: 'INLAND'

In [ ]:
model  = LinearRegression()

### First Iteration (Dropna + Onehot encoding of ocean)

In [ ]:
first_iteration = evaluation(model)

### Second Iteration : First iteration + log scale of skewed columns

In [ ]:
second_iteration = evaluation(model)

### Third Iteration : Second Iteration + StandardScaler

In [ ]:
third_iteration = evaluation(model)

## KNN

In [ ]:
model2 = KNeighborsRegressor(n_neighbors = 2)

In [ ]:
fourth_iteration = evaluation(model2)

In [ ]:
import pickle

with open("model.pkl", "wb") as file:
    pickle.dump(model2, file)